In [2]:
import pandas as pd

In [19]:
#Q1
#Load and inspect the df 
df = pd.read_csv("D:/Network Operations Predictive System/data/sms-call-internet-mi-2013-11-01.csv")
print(df.head())

              datetime  CellID  countrycode   smsin  smsout  callin  callout  \
0  2013-11-01 00:00:00       1            0  0.3521     NaN     NaN   0.0273   
1  2013-11-01 00:00:00       1           33     NaN     NaN     NaN      NaN   
2  2013-11-01 00:00:00       1           39  1.7322  1.1047  0.5919   0.4020   
3  2013-11-01 00:00:00       2            0  0.3581     NaN     NaN   0.0273   
4  2013-11-01 00:00:00       2           33     NaN     NaN     NaN      NaN   

   internet  
0       NaN  
1    0.0261  
2   57.7729  
3       NaN  
4    0.0274  


In [7]:
#shape
print("="*40)
print("SHAPE")
print("="*40)
print(df.shape,"\n")

#columns
print("="*40)
print("COLUMNS")
print("="*40)
print(df.columns,"\n")

#Dtype
print("="*40)
print("DTYPES")
print("="*40)
print(df.dtypes,"\n")

SHAPE
(1891928, 8) 

COLUMNS
Index(['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin',
       'callout', 'internet'],
      dtype='str') 

DTYPES
datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object 



In [8]:
#Q2
#raw-to-canonical mapping
column_mapping = {
    "datetime": "timestamp",
    "CellID" : "grid_id",
    "countrycode":"country_code",
    "smsin":"sms_in",
    "smsout":"sms_out",
    "callin":"call_in",
    "callout":"call_out",
    "internet":"internet_activity"
}
df = df.rename(columns = column_mapping)


In [9]:
#Q3
df["timestamp"] = pd.to_datetime(df["timestamp"])
timestamps = df["timestamp"].drop_duplicates().sort_values()
assert len(timestamps) == 24, "Timestamp count is not 24"
print("Distinct timestamps:", len(timestamps))

Distinct timestamps: 24


In [10]:
time_diff = timestamps.diff().dropna()
assert (time_diff == pd.Timedelta(hours=1)).all(),"Timestamps are not exactly one hour apart"
print("Time range:", timestamps.min(), "to", timestamps.max())

Time range: 2013-11-01 00:00:00 to 2013-11-01 23:00:00


In [11]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

In [12]:
#Q4
missing_grid_id = df["grid_id"].isna().sum()
missing_timestamp = df["timestamp"].isna().sum()
activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]
blank_activity = df[activity_columns].isna().sum()
exact_duplicates = df.duplicated().sum()
negative_activity = (df[activity_columns] < 0).sum()
print("Missing grid_id:", missing_grid_id)
print("Missing timestamp:", missing_timestamp)
print("\nBlank activity values:")
print(blank_activity)
print("\nExact duplicate rows:", exact_duplicates)
print("\nNegative activity values:")
print(negative_activity)

Missing grid_id: 0
Missing timestamp: 0

Blank activity values:
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
dtype: int64

Exact duplicate rows: 0

Negative activity values:
sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
dtype: int64


In [13]:
#Q5
grid_hour_counts = (
    df.groupby(["timestamp", "grid_id"])["country_code"]
      .nunique()
      .reset_index(name="country_code_count")
)
print(grid_hour_counts.head())
print("Maximum country-code rows per grid/hour:",grid_hour_counts["country_code_count"].max())
raw_duplicates = df.duplicated(
    subset=["timestamp", "grid_id", "country_code"]
).sum()

print("Duplicate raw-grain rows:", raw_duplicates)

   timestamp  grid_id  country_code_count
0 2013-11-01        1                   3
1 2013-11-01        2                   3
2 2013-11-01        3                   3
3 2013-11-01        4                   3
4 2013-11-01        5                   3
Maximum country-code rows per grid/hour: 65
Duplicate raw-grain rows: 0


In [14]:
#Q6
df["total_sms"] = df["sms_in"] + df["sms_out"]
df["total_calls"] = df["call_in"] + df["call_out"]
df["total_activity"] = (df["total_sms"] + df["total_calls"] + df["internet_activity"])

In [15]:
#Q7
unique_grids = df["grid_id"].nunique()
country_codes = sorted(df["country_code"].dropna().unique())
hourly_activity = (
    df.groupby("hour")["total_activity"]
      .sum()
      .sort_values(ascending=False)
)
grid_activity = (
    df.groupby("grid_id")["total_activity"]
      .sum()
      .sort_values(ascending=False)
)
null_counts = df.isna().sum()
print("="*40)
print("PROFILING FACTS")
print("="*40)
print("Unique grids:", unique_grids)
print("Time range:", timestamps.min(), "to", timestamps.max())
print("Cadence: 1 hour")
print("Number of country-code categories:", len(country_codes))
print("Busiest hour:", hourly_activity.index[0])
print("Busiest hour activity:", hourly_activity.iloc[0])
print("Busiest grid:", grid_activity.index[0])
print("Busiest grid activity:", grid_activity.iloc[0])
print("\nNull counts per column:")
print(null_counts)

PROFILING FACTS
Unique grids: 10000
Time range: 2013-11-01 00:00:00 to 2013-11-01 23:00:00
Cadence: 1 hour
Number of country-code categories: 246
Busiest hour: 11
Busiest hour activity: 5009588.4424
Busiest grid: 5161
Busiest grid activity: 262060.34

Null counts per column:
timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
total_sms            1502892
total_calls          1594716
total_activity       1658462
dtype: int64


In [16]:
# NP1 Acceptance Criteria Check

# 1. Exactly 24 distinct timestamps, one hour apart
timestamps = df["timestamp"].drop_duplicates().sort_values()
time_diff = timestamps.diff().dropna()

check_1 = (
    len(timestamps) == 24
    and (time_diff == pd.Timedelta(hours=1)).all()
)

# 2. Every grid_id is between 1 and 10000
check_2 = df["grid_id"].between(1, 10000).all()

# 3. Same grid + timestamp appears with multiple country codes
country_counts = (
    df.groupby(["timestamp", "grid_id"])["country_code"]
      .nunique()
)

check_3 = (country_counts > 1).any()

# 4. Null counts recorded and activity blanks are non-zero
null_counts = df.isna().sum()

activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

activity_null_count = df[activity_columns].isna().sum().sum()

check_4 = activity_null_count > 0

# Final result
checks = {
    "24 timestamps + hourly cadence": check_1,
    "grid_id within 1-10000": check_2,
    "multiple country codes for same grid/hour": check_3,
    "activity nulls are non-zero": check_4
}

print("NP1 ACCEPTANCE CHECK")
print("-" * 40)

for check, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'} - {check}")

print("\nNull counts per column:")
print(null_counts)

print("\nOverall:",
      "PASS" if all(checks.values()) else "FAIL")

NP1 ACCEPTANCE CHECK
----------------------------------------
PASS - 24 timestamps + hourly cadence
PASS - grid_id within 1-10000
PASS - multiple country codes for same grid/hour
PASS - activity nulls are non-zero

Null counts per column:
timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
total_sms            1502892
total_calls          1594716
total_activity       1658462
dtype: int64

Overall: PASS


In [17]:
print(df.columns)
print(df.dtypes)

Index(['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in',
       'call_out', 'internet_activity', 'date', 'hour', 'day_of_week',
       'total_sms', 'total_calls', 'total_activity'],
      dtype='str')
timestamp            datetime64[us]
grid_id                       int64
country_code                  int64
sms_in                      float64
sms_out                     float64
call_in                     float64
call_out                    float64
internet_activity           float64
date                         object
hour                          int32
day_of_week                     str
total_sms                   float64
total_calls                 float64
total_activity              float64
dtype: object


In [18]:
df.head()

,timestamp,grid_id,country_code,sms_in,sms_out,call_in,call_out,internet_activity,date,hour,day_of_week,total_sms,total_calls,total_activity
0,2013-11-01,1,0,0.3521,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,NaN,NaN,NaN
1,2013-11-01,1,33,NaN,NaN,NaN,NaN,0.0261,2013-11-01,0,Friday,NaN,NaN,NaN
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729,2013-11-01,0,Friday,2.8369,0.9939,61.6037
3,2013-11-01,2,0,0.3581,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,NaN,NaN,NaN
4,2013-11-01,2,33,NaN,NaN,NaN,NaN,0.0274,2013-11-01,0,Friday,NaN,NaN,NaN
